# Expense Tracker — Kaggle Master Training

**Run the notebook from top to bottom.** This notebook trains the initial **classical-ML Expense Intelligence pipeline** on Kaggle without modifying Kaggle's system scientific packages.

The initial training run deliberately **does not train a transformer**. Transformer support remains optional in the repository for a later experiment only.

Current ML components:
- **Transaction category classifier:** word + character TF-IDF with Logistic Regression
- **Merchant similarity index:** character TF-IDF + nearest-neighbor retrieval for merchant normalization
- **Duplicate similarity index:** character TF-IDF + nearest-neighbor retrieval for likely duplicate transactions
- **Transaction anomaly model:** Isolation Forest, only when numeric transaction features such as amount are available
- **Spending forecast model:** HistGradientBoostingRegressor, only when date/timestamp and amount features are available

With the currently configured text classification datasets, the first three components are applicable; anomaly detection and spending forecasting are expected to report `not_applicable` unless those datasets provide the required numeric/date fields.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Yoge-2004/expense-tracker.git"
BRANCH = "feature/ml-expense-intelligence"
WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "expense-tracker"
ML = REPO / "ml"

OUTPUT = WORK_ROOT / "expense-ml-runs"
DATA_CACHE = WORK_ROOT / "expense-ml-data"
HF_CACHE = Path("/kaggle/temp/huggingface")
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE / "transformers")
os.environ["MPLBACKEND"] = "Agg"
os.environ["EXPENSE_ML_OUTPUT"] = str(OUTPUT)
os.environ["EXPENSE_ML_DATA_DIR"] = str(DATA_CACHE)
os.environ["EXPENSE_ML_DATA_CACHE"] = str(DATA_CACHE)

def run(*args, cwd=None, env=None):
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    print("$", " ".join(map(str, args)))
    return subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, env=merged, check=True)

if REPO.exists():
    shutil.rmtree(REPO)
run("git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, REPO)
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Checked-out commit:", commit)
print("ML directory:", ML)

## 1. Prepare Kaggle authentication

This initial classical-ML pipeline does **not require a GPU**. A Kaggle GPU may be enabled, but it is not used for the initial classifier or auxiliary models.

Create a Kaggle Secret named `HF_TOKEN` when the configured Hugging Face datasets require authentication. The token is used only for dataset access and is never printed.


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token.strip()
except Exception as exc:
    print("HF_TOKEN secret lookup skipped:", type(exc).__name__)

print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))


## 2. Create the isolated Python 3.14 classical-ML environment

Kaggle's notebook kernel can remain on Python 3.12. The ML project is installed into its own `uv` environment using the Python version declared by the repository.

This initial run installs only the **classical training dependencies**. It does **not** install PyTorch, Transformers, Accelerate, or other transformer-training dependencies.


In [ ]:
if shutil.which("uv") is None:
    run(sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "uv")

UV = shutil.which("uv")
if UV is None:
    candidates = [Path(sys.prefix) / "bin" / "uv", Path.home() / ".local" / "bin" / "uv"]
    UV = next((str(p) for p in candidates if p.exists()), None)
if UV is None:
    raise FileNotFoundError("uv could not be located after installation.")

UV_CACHE = WORK_ROOT / "uv-cache"
UV_CACHE.mkdir(parents=True, exist_ok=True)

run(UV, "python", "install", "3.14")
run(UV, "sync", "--extra", "classical", "--extra", "dev", cwd=ML, env={"UV_CACHE_DIR": str(UV_CACHE)})

version = subprocess.check_output([UV, "run", "python", "--version"], cwd=ML, text=True).strip()
print("Project interpreter:", version)
if "3.14" not in version:
    raise RuntimeError(f"uv selected the wrong Python interpreter: {version}")

In [ ]:
preflight = r'''
import sys
import numpy, pandas, scipy, sklearn, datasets, matplotlib, tqdm

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Datasets:", datasets.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Matplotlib backend:", matplotlib.get_backend())
print("tqdm:", tqdm.__version__)
'''
run(
    UV, "run", "python", "-c", preflight,
    cwd=ML,
    env={
        "HF_HOME": os.environ["HF_HOME"],
        "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"],
        "TRANSFORMERS_CACHE": os.environ["TRANSFORMERS_CACHE"],
        "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
        "MPLBACKEND": "Agg",
    },
)


## 3. Start from a completely fresh training dataset/cache

The initial run must not silently reuse prepared data or source caches from an earlier experiment. Only disposable Kaggle working directories are cleared.

In [ ]:
for path in (DATA_CACHE, OUTPUT):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
print("Fresh initial-training workspace ready.")

## 4. Classical-ML training settings

These settings affect this Kaggle run only. They do not modify the production defaults in the repository.


In [ ]:
cpu_count = os.cpu_count() or 4
KAGGLE_ENV = {
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    "HF_HOME": os.environ["HF_HOME"],
    "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"],
    "TRANSFORMERS_CACHE": os.environ["TRANSFORMERS_CACHE"],
    "MPLBACKEND": "Agg",
    "EXPENSE_ML_DATA_DIR": str(DATA_CACHE),
    "EXPENSE_ML_OUTPUT": str(OUTPUT),
    "EXPENSE_ML_CPU_THREADS": str(min(cpu_count, 8)),
    "EXPENSE_ML_BASELINE_MAX_ROWS": "1500000",
    "EXPENSE_ML_MAX_MERCHANTS": "250000",
    "EXPENSE_ML_DUPLICATE_MAX_ROWS": "500000",
}
print(json.dumps({k: v for k, v in KAGGLE_ENV.items() if k != "HF_TOKEN"}, indent=2))


## 5. Run the complete initial classical-ML pipeline

The only category classifier trained in this initial run is the **TF-IDF + Logistic Regression** model.

The other applicable components are similarity/retrieval indexes for merchant normalization and duplicate detection. Anomaly detection and spending forecasting run only when the dataset schema contains the required features.

**Transformer training is intentionally disabled for this initial run.**


In [ ]:
prepared = DATA_CACHE / "transactions.parquet"
config = ML / "config" / "datasets.yaml"
run(
    UV, "run", "python", "-m", "expense_ml.master_pipeline",
    "--config", str(config),
    "--prepared", str(prepared),
    "--output", str(OUTPUT),
    cwd=ML,
    env=KAGGLE_ENV,
)

## 6. Inspect the completed run

The notebook stops with an error if the master pipeline did not finish successfully. The expected initial category model is `category_tfidf`; transformer status should be `disabled`.

In [ ]:
run_dirs = sorted(path for path in OUTPUT.iterdir() if path.is_dir() and (path / "manifest.json").exists())
if not run_dirs:
    raise RuntimeError("No master-training run with manifest.json was produced.")

RUN_DIR = run_dirs[-1]
MANIFEST = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
print("Run:", RUN_DIR.name)
print("Status:", MANIFEST.get("status"))
print("Pipeline:", MANIFEST.get("pipeline_version"))
print("\nSelected model:")
print(json.dumps(MANIFEST.get("selected_model"), indent=2))
print("\nFinal test:")
print(json.dumps(MANIFEST.get("test_evaluation"), indent=2))
if MANIFEST.get("india_holdout_evaluation"):
    print("\nIndia holdout:")
    print(json.dumps(MANIFEST["india_holdout_evaluation"], indent=2))
print("\nModel statuses:")
for name, details in MANIFEST.get("models", {}).items():
    print(f"  {name}: {details.get('status')}")

if MANIFEST.get("status") != "completed":
    raise RuntimeError("Master pipeline did not finish with status=completed.")

In [ ]:
reports = [
    "reports/dataset_summary.json",
    "reports/dataset_quality.json",
    "reports/split_summary.json",
    "reports/training_sampling.json",
    "reports/model_selection_validation.json",
    "reports/model_comparison.json",
    "reports/category_test.json",
    "reports/category_test_country_metrics.json",
    "reports/category_india_holdout.json",
    "reports/duplicate_candidates.json",
    "reports/anomaly_report.json",
    "reports/spending_forecast.json",
]

for relative in reports:
    path = RUN_DIR / relative
    if path.exists():
        print(f"\n===== {relative} =====")
        print(path.read_text(encoding="utf-8")[:12000])

## 7. Package the initial model artifacts

The archive contains the complete master run: trained models, manifests, reports, and figures. Raw datasets and Hugging Face caches are not included.

In [ ]:
archive_base = WORK_ROOT / f"expense-tracker-ml-initial-{RUN_DIR.name}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print("Kaggle artifact:", archive_path)
print("Artifact size MiB:", round(archive_path.stat().st_size / 1024**2, 2))
print("Run directory:", RUN_DIR)